# RSNA: How Failed Predictions Can Pass Your Checks

In this competition your notebook is rerun on ~1,300 hidden studies and you never see its logs. If a reader fails there and your loop quietly writes a fallback `0.5`, the submission still has the right rows, the right columns and values in [0, 1]. A CSV check cannot tell. This notebook **injects 15 such failures and 6 valid controls, and measures which checks notice**. It also gives you the checker.

**TL;DR**
- **The counterexample:** ranking by argsort-of-argsort turns 35 identical fallback rows into **35 distinct ranks**, so a "no constant columns" guard passes. Ranking with `rank(method="average")` keeps the ties, and the guard fires. With legitimately tied outputs, argsort-of-argsort also made **92.5% of cells change when only the input row order changed** (1,300 studies).
- **The measurement:** a CSV check flagged **4 of 15** injected faults (flagged = ERROR or WARN). Adding the raw pre-rank predictions raised that to **12 of 15**, and adding a per-study inference log to **14 of 15**, all as ERRORs. No layer raised an ERROR on any of the 6 valid controls.
- **The limit:** a **single** failed study is invisible to every value-based check under every rank step tested. Only a recorded failure shows it. A prediction read from the wrong study is caught only when the adapter records the header `StudyInstanceUID`.
- **Reusable:** `rsna_preflight.py` (CPU, no model weights) plus a reader/predictor adapter. It writes structured JSON diagnostics.

<sub>Scope: synthetic fixtures test bookkeeping. Real training DICOMs, with reports and labels excluded, test the reader. Nothing here measures AUC, GPU memory or hidden-test runtime, or guarantees that a hidden rerun succeeds.</sub>

In [ ]:
%%writefile rsna_preflight.py
"""rsna_preflight: catch inference failures that a valid-looking submission CSV hides.

Four evidence layers, each optional except the first:

    csv         the submission file alone (schema, IDs, finite values)
    guard       the common "every column is non-constant" assertion
    raw         the last prediction array BEFORE any rank/normalisation step,
                keyed by study ID and named with the model's own label order
    provenance  one record per study written while inference ran: status,
                fallback flag, images read, the StudyInstanceUID seen in the headers
    inputs      the input folder compared with the study/series lists

A CSV alone cannot show that a prediction belongs to the wrong study or label.
Those checks run only when the adapter records the evidence (raw labels, header
UIDs); otherwise the report says they were not verifiable.

Severity: ERROR = recorded contract failure; WARNING = suspicious but can be
legitimate (e.g. constant predictions); INFO = context.

Usage (library):
    rep = preflight(sub, expected_ids, raw=raw, provenance=prov)
    print(rep.summary()); rep.to_json("preflight_report.json")
Usage (command line):
    python rsna_preflight.py --submission submission.csv --expected test.csv \
        --raw raw_predictions.csv --provenance provenance.csv --json report.json
Exit code 1 when any ERROR is found.
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence

import numpy as np
import pandas as pd

__version__ = "1.0.0"

ID_COL = "StudyInstanceUID"
SERIES_COL = "SeriesInstanceUID"
# competition order, as in sample_submission.csv (checked 2026-10-10)
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
          "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
ERROR, WARNING, INFO = "ERROR", "WARNING", "INFO"
LAYERS = ("csv", "guard", "raw", "provenance", "inputs")


# --------------------------------------------------------------------------- report
@dataclass
class Finding:
    code: str
    severity: str
    layer: str
    message: str
    count: int = 0
    examples: list = field(default_factory=list)


@dataclass
class Report:
    findings: list[Finding] = field(default_factory=list)
    layers_run: list[str] = field(default_factory=list)
    meta: dict = field(default_factory=dict)

    def add(self, code, severity, layer, message, count=0, examples=()):
        self.findings.append(Finding(code, severity, layer, message, int(count),
                                     [str(x) for x in list(examples)[:5]]))

    def extend(self, other: "Report"):
        self.findings += other.findings
        self.layers_run += [x for x in other.layers_run if x not in self.layers_run]
        return self

    def by(self, severity):
        return [f for f in self.findings if f.severity == severity]

    @property
    def errors(self):
        return self.by(ERROR)

    @property
    def warnings(self):
        return self.by(WARNING)

    @property
    def status(self):
        return "FAIL" if self.errors else ("WARN" if self.warnings else "PASS")

    def codes(self, severity=None):
        return sorted({f.code for f in self.findings if severity is None or f.severity == severity})

    def table(self) -> pd.DataFrame:
        cols = ["severity", "layer", "code", "count", "message", "examples"]
        if not self.findings:
            return pd.DataFrame(columns=cols)
        order = {ERROR: 0, WARNING: 1, INFO: 2}
        t = pd.DataFrame([asdict(f) for f in self.findings])[cols]
        return t.sort_values("severity", key=lambda s: s.map(order), kind="stable").reset_index(drop=True)

    def summary(self):
        return (f"preflight {self.status}: {len(self.errors)} error(s), {len(self.warnings)} warning(s); "
                f"layers run: {', '.join(self.layers_run) or 'none'}")

    def to_dict(self):
        return {"tool": "rsna_preflight", "version": __version__, "status": self.status,
                "layers_run": self.layers_run, "meta": self.meta,
                "findings": [asdict(f) for f in self.findings]}

    def to_json(self, path=None):
        s = json.dumps(self.to_dict(), indent=2, default=str)
        if path:
            Path(path).write_text(s, encoding="utf-8")
        return s


# --------------------------------------------------------------------------- helpers
def _ids(values) -> pd.Series:
    return pd.Series(values).astype(str).str.strip()


def _labels(sub: pd.DataFrame, labels, id_col):
    return list(labels) if labels else [c for c in sub.columns if c != id_col]


def default_labels(sample_submission: pd.DataFrame, id_col=ID_COL):
    """The label columns, in competition order, taken from sample_submission.csv."""
    return [c for c in sample_submission.columns if c != id_col]


# --------------------------------------------------------------------------- layer: csv
def check_csv(sub: pd.DataFrame, expected_ids: Iterable, labels: Sequence[str] = LABELS,
              id_col: str = ID_COL) -> Report:
    """Baseline: what a plain submission validator checks."""
    rep = Report(layers_run=["csv"])
    labels = _labels(sub, labels, id_col)
    if id_col not in sub.columns:
        rep.add("CSV_NO_ID_COLUMN", ERROR, "csv", f"column '{id_col}' is missing")
        return rep
    missing_cols = [c for c in labels if c not in sub.columns]
    extra_cols = [c for c in sub.columns if c != id_col and c not in labels]
    if missing_cols:
        rep.add("CSV_MISSING_LABEL_COLUMNS", ERROR, "csv", "label columns missing", len(missing_cols), missing_cols)
    if extra_cols:
        rep.add("CSV_EXTRA_COLUMNS", WARNING, "csv", "unexpected extra columns", len(extra_cols), extra_cols)
    ids, exp = _ids(sub[id_col]), _ids(list(expected_ids))
    exp_set, id_set = set(exp), set(ids)
    dup = ids[ids.duplicated()].unique()
    if len(dup):
        rep.add("CSV_DUPLICATE_IDS", ERROR, "csv", "study IDs appear more than once", len(dup), dup)
    miss = sorted(exp_set - id_set)
    if miss:
        rep.add("CSV_MISSING_IDS", ERROR, "csv", "expected studies have no row", len(miss), miss)
    foreign = sorted(id_set - exp_set)
    if foreign:
        rep.add("CSV_FOREIGN_IDS", ERROR, "csv", "rows for studies not in the expected list", len(foreign), foreign)
    present = [c for c in labels if c in sub.columns]
    vals = sub[present].apply(pd.to_numeric, errors="coerce")
    nonnum = int((vals.isna() & sub[present].notna()).sum().sum())
    if nonnum:
        rep.add("CSV_NON_NUMERIC", ERROR, "csv", "non-numeric prediction cells", nonnum)
    arr = vals.to_numpy(dtype=float)
    bad = ~np.isfinite(arr)
    if bad.any():
        rows = sub.loc[bad.any(axis=1), id_col].astype(str)
        rep.add("CSV_NONFINITE", ERROR, "csv", "NaN/Inf/empty prediction cells", int(bad.sum()), rows)
    finite = arr[np.isfinite(arr)]
    if finite.size and (finite.min() < 0 or finite.max() > 1):
        rep.add("CSV_OUT_OF_RANGE", WARNING, "csv",
                f"values outside [0, 1] (min {finite.min():.4g}, max {finite.max():.4g}); confirm the expected format",
                int(((finite < 0) | (finite > 1)).sum()))
    if not len(dup) and not miss and not foreign and list(ids) != list(exp):
        rep.add("CSV_ROW_ORDER", INFO, "csv",
                "row order differs from the expected list; rows are keyed by study ID, so this alone is not an error")
    return rep


# --------------------------------------------------------------------------- layer: guard
def check_nonconstant(sub: pd.DataFrame, labels: Sequence[str] = LABELS, id_col: str = ID_COL) -> Report:
    """The common guard 'each prediction column has more than one distinct value', plus
    'no two studies share an identical row'.

    Both are WARNINGs: constant or tied predictions can be legitimate. Applied AFTER
    a tie-breaking rank step (argsort of argsort) neither can ever fire."""
    rep = Report(layers_run=["guard"])
    labels = [c for c in _labels(sub, labels, id_col) if c in sub.columns]
    vals = sub[labels].apply(pd.to_numeric, errors="coerce")
    const = [c for c in labels if vals[c].nunique(dropna=False) <= 1]
    if const:
        rep.add("GUARD_CONSTANT_COLUMNS", WARNING, "guard", "prediction columns with a single value",
                len(const), const)
    if len(vals) >= 2 and not const:
        dup = vals.duplicated(keep=False)
        if dup.any():
            ids = sub.loc[dup, id_col] if id_col in sub.columns else dup[dup].index
            rep.add("GUARD_IDENTICAL_ROWS", WARNING, "guard",
                    f"{int(dup.sum())} studies share an identical {len(labels)}-value row (possible fallback)",
                    int(dup.sum()), ids)
    return rep


# --------------------------------------------------------------------------- layer: raw
def _tie_and_order(raw_v: np.ndarray, fin_v: np.ndarray, tol=1e-9):
    """Rows in raw-tied groups that received different final values, and the number of
    adjacent raw-ordered groups whose final values go backwards."""
    order = np.argsort(raw_v, kind="stable")
    r, f = raw_v[order], fin_v[order]
    starts = np.r_[0, np.flatnonzero(np.diff(r) != 0) + 1]
    gmin = np.minimum.reduceat(f, starts)
    gmax = np.maximum.reduceat(f, starts)
    sizes = np.diff(np.r_[starts, len(r)])
    broken = int(sizes[(gmax - gmin) > tol].sum())
    backwards = int((gmax[:-1] > gmin[1:] + tol).sum()) if len(starts) > 1 else 0
    return broken, backwards


def _spearman(a, b):
    ra, rb = pd.Series(a).rank().to_numpy(), pd.Series(b).rank().to_numpy()
    if ra.std() == 0 or rb.std() == 0:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])


def check_raw(raw: pd.DataFrame, sub: pd.DataFrame, labels: Sequence[str] = LABELS, id_col: str = ID_COL,
              fallback: Sequence[float] | float | None = None) -> Report:
    """Compare the submission with the raw (pre-normalisation) predictions.

    Assumes the submission is a per-label monotonic transform of `raw` (identity,
    probability calibration, rank, rank-percentile). If anything is blended in after
    `raw` is captured, capture `raw` later."""
    rep = Report(layers_run=["raw"])
    labels = [c for c in _labels(sub, labels, id_col) if c in sub.columns]
    if id_col not in raw.columns:
        rep.add("RAW_NO_ID_COLUMN", ERROR, "raw", f"raw predictions lack '{id_col}'")
        return rep
    raw = raw.copy()
    raw[id_col] = _ids(raw[id_col])
    dup = raw[id_col][raw[id_col].duplicated()].unique()
    if len(dup):
        rep.add("RAW_DUPLICATE_IDS", ERROR, "raw", "studies predicted more than once", len(dup), dup)
    raw_labels = [c for c in raw.columns if c != id_col]
    if set(raw_labels) != set(labels):
        rep.add("RAW_LABEL_SET_MISMATCH", ERROR, "raw", "raw label names differ from submission columns",
                len(set(raw_labels) ^ set(labels)), sorted(set(raw_labels) ^ set(labels)))
    sub_ids = set(_ids(sub[id_col])) if id_col in sub.columns else set()
    no_raw = sorted(sub_ids - set(raw[id_col]))
    if no_raw:
        rep.add("RAW_MISSING_FOR_SUBMITTED", ERROR, "raw",
                "submission rows with no recorded model prediction (partial run or pre-filled default)",
                len(no_raw), no_raw)
    extra = sorted(set(raw[id_col]) - sub_ids)
    if extra:
        rep.add("RAW_NOT_SUBMITTED", WARNING, "raw", "predicted studies absent from the submission", len(extra), extra)
    rv = raw[raw_labels].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    nf = ~np.isfinite(rv)
    if nf.any():
        rep.add("RAW_NONFINITE", ERROR, "raw", "NaN/Inf in raw predictions (the CSV may have been sanitised)",
                int(nf.sum()), raw.loc[nf.any(axis=1), id_col])
    ok_rows = ~nf.any(axis=1)
    if ok_rows.sum() >= 2:
        keys = pd.Series([v.tobytes() for v in rv[ok_rows]])
        counts = keys.map(keys.value_counts())
        shared = int((counts > 1).sum())
        if shared == ok_rows.sum():
            rep.add("RAW_ALL_IDENTICAL", WARNING, "raw",
                    "every study has the same prediction vector (constant output; can be legitimate)", shared)
        elif shared:
            rep.add("RAW_IDENTICAL_VECTORS", WARNING, "raw",
                    f"{shared} studies share an exact {len(raw_labels)}-value prediction vector with another study "
                    f"(typical of a fallback; largest group {int(counts.max())})", shared,
                    raw.loc[ok_rows, id_col].to_numpy()[(counts > 1).to_numpy()])
    if fallback is not None:
        fb = np.broadcast_to(np.asarray(fallback, dtype=float), (len(raw_labels),))
        hit = np.all(rv == fb, axis=1)
        if hit.any():
            rep.add("RAW_EQUALS_FALLBACK", WARNING, "raw", "raw predictions equal the declared fallback vector",
                    int(hit.sum()), raw.loc[hit, id_col])
    # submission vs raw, row by row
    if id_col not in sub.columns or not labels:
        return rep
    s = sub.copy()
    s[id_col] = _ids(s[id_col])
    m = raw.drop_duplicates(id_col).merge(s.drop_duplicates(id_col), on=id_col, suffixes=("__raw", "__sub"))
    if len(m) < 2:
        return rep
    tie_rows, tie_labels, misaligned = 0, [], []
    for c in labels:
        if c not in raw_labels:
            continue
        a = pd.to_numeric(m[f"{c}__raw"], errors="coerce").to_numpy(dtype=float)
        b = pd.to_numeric(m[f"{c}__sub"], errors="coerce").to_numpy(dtype=float)
        good = np.isfinite(a) & np.isfinite(b)
        if good.sum() < 2:
            continue
        broken, backwards = _tie_and_order(a[good], b[good])
        if broken:
            tie_rows += broken
            tie_labels.append(c)
        if backwards:
            best = max(((_spearman(pd.to_numeric(m[f"{o}__raw"], errors="coerce"), b), o)
                        for o in raw_labels if o != c), default=(float("nan"), None))
            hint = f"{c} (submitted values follow raw '{best[1]}', rho={best[0]:.3f})" if best[0] > 0.999 else c
            misaligned.append(hint)
    if tie_labels:
        rep.add("RAW_TIES_BROKEN", WARNING, "raw",
                "normalisation gave different values to studies whose raw predictions were tied "
                "(tie-breaking rank; hides constant/fallback rows from value-based checks)",
                tie_rows, tie_labels)
    if misaligned:
        rep.add("RAW_ORDER_MISMATCH", ERROR, "raw",
                "submitted values are not a monotonic function of the raw predictions: rows or labels were "
                "reassigned between prediction and export", len(misaligned), misaligned)
    return rep


# --------------------------------------------------------------------------- layer: provenance
PROVENANCE_COLUMNS = ["study_id", "status", "fallback", "n_series", "n_images",
                      "observed_study_uid", "error", "seconds"]


def check_provenance(prov: pd.DataFrame, expected_ids: Iterable, id_col: str = ID_COL,
                     max_fallback: int = 0) -> Report:
    """Per-study inference records written while the pipeline ran (see ProvenanceLog)."""
    rep = Report(layers_run=["provenance"])
    if "study_id" not in prov.columns:
        rep.add("PROV_NO_ID_COLUMN", ERROR, "provenance", "provenance lacks 'study_id'")
        return rep
    p = prov.copy()
    p["study_id"] = _ids(p["study_id"])
    exp = set(_ids(list(expected_ids)))
    missing = sorted(exp - set(p["study_id"]))
    if missing:
        rep.add("PROV_NO_RECORD", ERROR, "provenance", "expected studies with no inference record",
                len(missing), missing)
    dup = p["study_id"][p["study_id"].duplicated()].unique()
    if len(dup):
        rep.add("PROV_DUPLICATE_RECORDS", WARNING, "provenance", "studies recorded more than once", len(dup), dup)
    if "status" in p.columns:
        bad = p[p["status"].astype(str) != "ok"]
        if len(bad):
            errs = bad.get("error", pd.Series([""] * len(bad), index=bad.index)).fillna("").astype(str)
            ex = [f"{i}: {e.splitlines()[0][:120] if e else s}" for i, e, s in
                  zip(bad["study_id"], errs, bad["status"])]
            rep.add("PROV_STUDY_FAILED", ERROR, "provenance", "studies whose reader or predictor failed",
                    len(bad), ex)
    if "fallback" in p.columns:
        fb = p[p["fallback"].fillna(False).astype(bool)]
        if len(fb) > max_fallback:
            rep.add("PROV_FALLBACK_USED", ERROR, "provenance",
                    f"fallback predictions written for {len(fb)} studies (allowed: {max_fallback})", len(fb),
                    fb["study_id"])
    if "n_images" in p.columns:
        ok = p["status"].astype(str) == "ok" if "status" in p.columns else pd.Series(True, index=p.index)
        z = p[ok & (pd.to_numeric(p["n_images"], errors="coerce").fillna(0) <= 0)]
        if len(z):
            rep.add("PROV_NO_IMAGES", ERROR, "provenance", "studies reported ok but predicted from zero images",
                    len(z), z["study_id"])
    uid = p.get("observed_study_uid")
    if uid is None or uid.isna().all() or (uid.astype(str).str.strip() == "").all():
        rep.add("PROV_ASSOCIATION_UNVERIFIED", INFO, "provenance",
                "no header StudyInstanceUID recorded; prediction-to-study association cannot be checked")
    else:
        seen = p[uid.notna() & (uid.astype(str).str.strip() != "")]
        wrong = seen[_ids(seen["observed_study_uid"]).to_numpy() != seen["study_id"].to_numpy()]
        if len(wrong):
            rep.add("PROV_UID_MISMATCH", ERROR, "provenance",
                    "prediction keyed to one study was computed from another study's images", len(wrong),
                    [f"{a} <- {b}" for a, b in zip(wrong["study_id"], wrong["observed_study_uid"])])
    return rep


# --------------------------------------------------------------------------- layer: inputs
def check_inputs(series_root, study_ids: Iterable, series: pd.DataFrame | None = None,
                 study_col: str = ID_COL, series_col: str = SERIES_COL, ext=".dcm") -> tuple[Report, pd.DataFrame]:
    """Compare <series_root>/<study>/<series>/*.dcm with the study list (and series table)."""
    rep = Report(layers_run=["inputs"])
    root = Path(series_root)
    rows = []
    listed = {}
    if series is not None and study_col in series.columns and series_col in series.columns:
        for s, g in series.astype({study_col: str, series_col: str}).groupby(study_col):
            listed[s] = list(g[series_col])
    for s in _ids(list(study_ids)):
        sd = root / s
        names = listed.get(s) or (sorted(os.listdir(sd)) if sd.is_dir() else [])
        for se in names:
            d = sd / se
            n = len([x for x in os.listdir(d) if x.endswith(ext)]) if d.is_dir() else -1
            rows.append({"study_id": s, "series_id": se, "n_files": n})
        if not names:
            rows.append({"study_id": s, "series_id": None, "n_files": -1})
    t = pd.DataFrame(rows, columns=["study_id", "series_id", "n_files"])
    nostudy = t[(t["series_id"].isna())]["study_id"]
    if len(nostudy):
        rep.add("INPUT_STUDY_MISSING", ERROR, "inputs", "study folder missing or empty", len(nostudy), nostudy)
    gone = t[t["series_id"].notna() & (t["n_files"] < 0)]
    if len(gone):
        rep.add("INPUT_SERIES_MISSING", ERROR, "inputs", "listed series folder missing", len(gone),
                gone["study_id"] + "/" + gone["series_id"])
    empty = t[t["series_id"].notna() & (t["n_files"] == 0)]
    if len(empty):
        rep.add("INPUT_SERIES_EMPTY", ERROR, "inputs", f"series folder has no {ext} files", len(empty),
                empty["study_id"] + "/" + empty["series_id"])
    return rep, t


# --------------------------------------------------------------------------- adapter + runner
@dataclass
class StudyData:
    """What a reader returns. `images` is whatever your predictor consumes."""
    images: Any
    n_series: int
    n_images: int
    observed_study_uid: str | None = None


class ProvenanceLog:
    """Two-line integration for an existing loop:  log.record(study_id, status='ok', ...)."""

    def __init__(self):
        self.rows = []

    def record(self, study_id, status="ok", fallback=False, n_series=None, n_images=None,
               observed_study_uid=None, error=None, seconds=None):
        self.rows.append(dict(study_id=str(study_id), status=status, fallback=bool(fallback),
                              n_series=n_series, n_images=n_images, observed_study_uid=observed_study_uid,
                              error=error, seconds=seconds))

    def frame(self):
        return pd.DataFrame(self.rows, columns=PROVENANCE_COLUMNS)


@dataclass
class InferenceResult:
    raw: pd.DataFrame
    provenance: pd.DataFrame
    submission: pd.DataFrame


def run_inference(study_ids: Sequence, read_study: Callable[[str], StudyData],
                  predict: Callable[[StudyData], Sequence[float]], labels: Sequence[str] = LABELS,
                  fallback: Sequence[float] | float = 0.5, normalize: Callable | None = None,
                  id_col: str = ID_COL) -> InferenceResult:
    """Reference loop: read -> predict -> record, with an explicit fallback that is logged.

    `predict` must return len(labels) values in the order of `labels` (the model's order).
    `normalize`, if given, maps the raw DataFrame (labels only) to final values."""
    labels = list(labels)
    fb = np.broadcast_to(np.asarray(fallback, dtype=float), (len(labels),)).copy()
    log, out = ProvenanceLog(), []
    for sid in [str(s) for s in study_ids]:
        t0, data = time.perf_counter(), None
        try:
            data = read_study(sid)
            y = np.asarray(predict(data), dtype=float).reshape(-1)
            if y.shape != (len(labels),):
                raise ValueError(f"predict returned shape {y.shape}, expected ({len(labels)},)")
            if not np.isfinite(y).all():
                raise FloatingPointError("predict returned NaN/Inf")
            status, err, used_fb = "ok", None, False
        except Exception as e:  # noqa: BLE001 - every failure is recorded, then the fallback is written
            y, status, err, used_fb = fb.copy(), "error", f"{type(e).__name__}: {e}", True
        log.record(sid, status=status, fallback=used_fb,
                   n_series=getattr(data, "n_series", None), n_images=getattr(data, "n_images", None),
                   observed_study_uid=getattr(data, "observed_study_uid", None), error=err,
                   seconds=round(time.perf_counter() - t0, 4))
        out.append(y)
    raw = pd.DataFrame(np.vstack(out) if out else np.empty((0, len(labels))), columns=labels)
    raw.insert(0, id_col, [str(s) for s in study_ids])
    final = raw.copy()
    if normalize is not None and len(raw):
        final[labels] = normalize(raw[labels])
    return InferenceResult(raw, log.frame(), final)


# --------------------------------------------------------------------------- normalisers
def rank_average(df: pd.DataFrame) -> pd.DataFrame:
    """Rank-percentile with ties averaged: equal inputs stay equal."""
    return df.rank(method="average", pct=True)


def rank_double_argsort(df: pd.DataFrame) -> pd.DataFrame:
    """argsort-of-argsort rank: every tie is broken by row position (shown as the anti-pattern)."""
    a = df.to_numpy(dtype=float)
    r = np.argsort(np.argsort(a, axis=0), axis=0)
    return pd.DataFrame((r + 1) / len(a), index=df.index, columns=df.columns)


# --------------------------------------------------------------------------- entry point
def preflight(submission: pd.DataFrame, expected_ids: Iterable, labels: Sequence[str] = LABELS,
              raw: pd.DataFrame | None = None, provenance: pd.DataFrame | None = None,
              inputs: Report | None = None, fallback=None, id_col: str = ID_COL,
              guard: bool = True, max_fallback: int = 0) -> Report:
    expected_ids = list(expected_ids)
    rep = Report(meta={"n_expected": len(expected_ids), "n_rows": int(len(submission))})
    rep.extend(check_csv(submission, expected_ids, labels, id_col))
    if guard:
        rep.extend(check_nonconstant(submission, labels, id_col))
    if raw is not None:
        rep.extend(check_raw(raw, submission, labels, id_col, fallback))
    if provenance is not None:
        rep.extend(check_provenance(provenance, expected_ids, id_col, max_fallback))
    if inputs is not None:
        rep.extend(inputs)
    for layer in ("raw", "provenance", "inputs"):
        if layer not in rep.layers_run:
            rep.add(f"{layer.upper()}_NOT_RUN", INFO, layer, f"{layer} evidence not supplied; its checks did not run")
    return rep


def _main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--submission", required=True)
    ap.add_argument("--expected", required=True, help="CSV with the expected study IDs (e.g. test.csv)")
    ap.add_argument("--id-col", default=ID_COL)
    ap.add_argument("--raw")
    ap.add_argument("--provenance")
    ap.add_argument("--fallback", type=float)
    ap.add_argument("--max-fallback", type=int, default=0)
    ap.add_argument("--json")
    a = ap.parse_args(argv)
    sub = pd.read_csv(a.submission)
    exp = pd.read_csv(a.expected)[a.id_col]
    rep = preflight(sub, exp, raw=pd.read_csv(a.raw) if a.raw else None,
                    provenance=pd.read_csv(a.provenance) if a.provenance else None,
                    fallback=a.fallback, id_col=a.id_col, max_fallback=a.max_fallback)
    with pd.option_context("display.max_colwidth", 100, "display.width", 200):
        print(rep.summary())
        print(rep.table().to_string(index=False))
    if a.json:
        rep.to_json(a.json)
    return 1 if rep.errors else 0


if __name__ == "__main__":
    sys.exit(_main())

In [ ]:
%%writefile preflight_lab.py
"""Fault-injection lab for rsna_preflight (synthetic bookkeeping fixtures).

A SyntheticSite is a small on-disk stand-in for test_series/: one placeholder file
per series holding the study UID and a codec tag. Its stub reader really opens those
files, so a missing folder is a real FileNotFoundError; a 'jpeg2000' tag raises the
way a pydicom read without a decoder plugin does (simulated). Real DICOMs are tested
separately. Nothing here is a model, loads weights, or measures GPU memory/runtime.
"""
from __future__ import annotations

import hashlib
import json
import os
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd

import rsna_preflight as rp

ID, LABELS = rp.ID_COL, rp.LABELS
METHODS = ["CSV check", "+ non-constant guard", "+ raw predictions", "+ provenance & inputs"]


def synthetic_ids(n, seed):
    rng = np.random.default_rng(seed)
    out = set()
    while len(out) < n:
        a, b = rng.integers(10**17, 10**18, size=2)
        out.add(f"1.2.826.0.1.3680043.8.498.{a}{b}")
    return sorted(out)


def _h(s):
    return int(hashlib.sha256(s.encode()).hexdigest()[:8], 16)


class SyntheticSite:
    def __init__(self, root, ids, n_series=3):
        self.root = Path(root)
        shutil.rmtree(self.root, ignore_errors=True)
        rows = []
        for sid in ids:
            for j in range(n_series):
                se = f"{sid}.{j + 1}"
                d = self.root / sid / se
                d.mkdir(parents=True)
                (d / "0.dcm").write_text(json.dumps({"study": sid, "codec": "raw"}))
                rows.append({ID: sid, rp.SERIES_COL: se})
        self.series = pd.DataFrame(rows)

    def set_codec(self, sids, codec):
        for sid in sids:
            for f in (self.root / sid).glob("*/0.dcm"):
                d = json.loads(f.read_text())
                d["codec"] = codec
                f.write_text(json.dumps(d))

    def reader(self, record_uid=True):
        listed = {s: list(g[rp.SERIES_COL]) for s, g in self.series.groupby(ID)}

        def read(sid):
            images, uids = {}, set()
            for se in listed[sid]:
                f = self.root / sid / se / "0.dcm"
                head = json.loads(f.read_text())                      # FileNotFoundError if missing
                if head["codec"] != "raw":
                    raise NotImplementedError(f"no pixel decoder for {head['codec']} (simulated)")
                uids.add(head["study"])
                images[se] = np.random.default_rng(_h(head["study"] + se[-2:])).random(16)
            return rp.StudyData(images, len(images), len(images),
                                uids.pop() if (record_uid and len(uids) == 1) else None)
        return read


W = np.random.default_rng(0).normal(size=(len(LABELS), 16))


def stub_predict(d: rp.StudyData):
    x = np.mean([v for v in d.images.values()], axis=0)
    return 1 / (1 + np.exp(-(W @ (x - 0.5)) * 2))


def user_loop(site, ids, *, normalize=rp.rank_average, predict=stub_predict, open_ids=None,
              record_uid=True, stop_after=None, nan_for=(), nan_fill=None, read=None):
    """A typical inference loop, instrumented with ProvenanceLog and a raw capture.

    open_ids: which study is actually opened for each key (models a path/order bug).
    stop_after: only the first k studies are processed (partial run); the rest of the
    submission keeps the 0.5 default it was pre-filled with."""
    read = read or site.reader(record_uid)
    log, raw = rp.ProvenanceLog(), []
    pairs = list(zip(ids, open_ids or ids))[:stop_after]
    for key, opened in pairs:
        try:
            d = read(opened)
            y = np.asarray(predict(d), dtype=float)
            if key in nan_for:
                y[0] = np.nan                                         # silent numerical failure
            log.record(key, n_series=d.n_series, n_images=d.n_images, observed_study_uid=d.observed_study_uid)
        except Exception as e:  # noqa: BLE001
            y = np.full(len(LABELS), 0.5)
            log.record(key, status="error", fallback=True, error=f"{type(e).__name__}: {e}")
        raw.append([key, *y])
    raw = pd.DataFrame(raw, columns=[ID, *LABELS])
    sub = pd.DataFrame({ID: ids}).merge(raw, on=ID, how="left")
    vals = sub[LABELS].fillna(0.5) if (stop_after is not None or nan_fill is not None) else sub[LABELS]
    sub[LABELS] = normalize(vals) if normalize else vals
    inputs, _ = rp.check_inputs(site.root, ids, site.series)       # audited in the same (faulty) state
    return {"sub": sub, "raw": raw, "prov": log.frame(), "inputs": inputs}


@dataclass
class Fixture:
    fid: str
    kind: str            # "fault" or "valid"
    name: str
    injected: str
    expect: str          # predeclared: first method expected to flag it, with severity, or "none"
    build: Callable


def _k(n, frac, lo):
    return int(min(max(lo, round(frac * n)), n - 1))


def fixtures():
    """Predeclared fixture list. Each build(site, ids) returns sub/raw/prov."""
    def clean(site, ids, **kw):
        return user_loop(site, ids, **kw)

    def mutate(fn, **kw):
        def b(site, ids):
            r = user_loop(site, ids, **kw)
            r["sub"] = fn(r["sub"].copy(), ids)
            return r
        return b

    def with_codec(count, **kw):
        def b(site, ids):
            bad = ids[:count(len(ids))]
            site.set_codec(bad, "jpeg2000")
            try:
                return user_loop(site, ids, **kw)
            finally:
                site.set_codec(bad, "raw")
        return b

    def missing_series(site, ids):
        d = site.root / ids[1] / f"{ids[1]}.2"
        tmp = d.with_name(d.name + ".hidden")
        d.rename(tmp)
        try:
            return user_loop(site, ids)
        finally:
            tmp.rename(d)

    def writer_shift(site, ids):
        r = user_loop(site, ids)
        s = r["sub"].copy()
        s[ID] = list(ids[1:]) + list(ids[:1])         # off-by-one when the ID column is attached
        r["sub"] = s
        return r

    def swap_labels(s, ids):
        s[["ACL", "MCL"]] = s[["MCL", "ACL"]].to_numpy()
        return s

    def foreign(s, ids):
        s.loc[1, ID] = "1.2.826.0.1.3680043.8.498.999"
        return s

    def quantised(d):
        return np.round(stub_predict(d), 2)

    def prior(d):
        return np.linspace(0.05, 0.6, len(LABELS))   # one fixed prior vector for every study

    rot = lambda ids: list(ids[1:]) + list(ids[:1])   # noqa: E731
    return [
        Fixture("F01", "fault", "missing study row", "one row dropped", "CSV check: ERROR",
                mutate(lambda s, i: s.drop(index=1).reset_index(drop=True))),
        Fixture("F02", "fault", "duplicate study row", "one row appended twice", "CSV check: ERROR",
                mutate(lambda s, i: pd.concat([s, s.iloc[[1]]], ignore_index=True))),
        Fixture("F03", "fault", "foreign study ID", "one ID replaced by an unknown UID", "CSV check: ERROR",
                mutate(foreign)),
        Fixture("F04", "fault", "NaN in the CSV", "one cell set to NaN at export", "CSV check: ERROR",
                mutate(lambda s, i: s.assign(ACL=s["ACL"].where(s.index != 1)))),
        Fixture("F05", "fault", "NaN hidden by fillna", "model returns NaN for one study; filled with 0.5 before ranking",
                "+ raw predictions: ERROR",
                lambda site, ids: user_loop(site, ids, nan_for=(ids[1],), nan_fill=0.5)),
        Fixture("F06", "fault", "partial run", "loop stops after 70% of studies; rest keep the pre-filled 0.5",
                "+ non-constant guard: WARNING",
                lambda site, ids: user_loop(site, ids, stop_after=_k(len(ids), 0.7, 1))),
        Fixture("F07", "fault", "missing series folder", "one listed series folder absent; reader fails, fallback written",
                "+ provenance & inputs: ERROR", missing_series),
        Fixture("F08", "fault", "reader failure, average rank", "decoder missing for ~3% of studies (min 2); fallback 0.5",
                "+ non-constant guard: WARNING", with_codec(lambda n: _k(n, 0.03, 2))),
        Fixture("F09", "fault", "partial fallback, argsort-of-argsort", "as F08, ranked by argsort of argsort",
                "+ raw predictions: WARNING", with_codec(lambda n: _k(n, 0.03, 2), normalize=rp.rank_double_argsort)),
        Fixture("F10", "fault", "all fallback, average rank", "every study fails; all 0.5", "+ non-constant guard: WARNING",
                with_codec(lambda n: n)),
        Fixture("F11", "fault", "all fallback, argsort-of-argsort", "every study fails; all 0.5, then argsort of argsort",
                "+ raw predictions: WARNING", with_codec(lambda n: n, normalize=rp.rank_double_argsort)),
        Fixture("F12", "fault", "rows shifted at export", "ID column attached off by one row", "+ raw predictions: ERROR",
                writer_shift),
        Fixture("F13", "fault", "label columns swapped", "ACL and MCL values exchanged at export",
                "+ raw predictions: ERROR", mutate(swap_labels)),
        Fixture("F14", "fault", "wrong study read (UID logged)", "loop opens the next study's folder; header UID recorded",
                "+ provenance & inputs: ERROR", lambda site, ids: user_loop(site, ids, open_ids=rot(ids))),
        Fixture("F15", "fault", "wrong study read (no UID)", "as F14, adapter records no header UID",
                "none (needs header evidence)",
                lambda site, ids: user_loop(site, ids, open_ids=rot(ids), record_uid=False)),
        Fixture("V01", "valid", "clean, average rank", "no fault", "none", clean),
        Fixture("V02", "valid", "clean, rows shuffled", "rows reordered, still keyed by UID", "none",
                mutate(lambda s, i: s.sample(frac=1, random_state=0).reset_index(drop=True))),
        Fixture("V03", "valid", "clean, raw probabilities", "no rank step", "none",
                lambda site, ids: user_loop(site, ids, normalize=None)),
        Fixture("V04", "valid", "constant prior, intended", "every study succeeds; model outputs one prior vector",
                "+ non-constant guard: WARNING (allowed)", lambda site, ids: user_loop(site, ids, predict=prior)),
        Fixture("V05", "valid", "rounded outputs, average rank", "probabilities rounded to 2 decimals (legit ties)",
                "none", lambda site, ids: user_loop(site, ids, predict=quantised)),
        Fixture("V06", "valid", "rounded outputs, argsort-of-argsort", "as V05, ranked by argsort of argsort",
                "+ raw predictions: WARNING (allowed)",
                lambda site, ids: user_loop(site, ids, predict=quantised, normalize=rp.rank_double_argsort)),
    ]


def method_reports(case, ids, site):
    sub, raw, prov, inputs = case["sub"], case["raw"], case["prov"], case["inputs"]
    return {
        METHODS[0]: rp.preflight(sub, ids, guard=False),
        METHODS[1]: rp.preflight(sub, ids),
        METHODS[2]: rp.preflight(sub, ids, raw=raw),
        METHODS[3]: rp.preflight(sub, ids, raw=raw, provenance=prov, inputs=inputs),
    }


def outcome(rep):
    return "ERROR" if rep.errors else ("WARN" if rep.warnings else "silent")


def run_matrix(site_root, n, seed, perm_seed=None, fx=None):
    """Run every fixture on an n-study site; returns long table and the reports."""
    ids = synthetic_ids(n, seed)
    if perm_seed is not None:
        ids = list(np.random.default_rng(perm_seed).permutation(ids))
    site = SyntheticSite(site_root, sorted(ids))
    rows, reps, cases = [], {}, {}
    for f in (fx or fixtures()):
        case = f.build(site, ids)
        r = method_reports(case, ids, site)
        reps[f.fid], cases[f.fid] = r, case
        for m, rep in r.items():
            rows.append({"fid": f.fid, "kind": f.kind, "name": f.name, "method": m, "outcome": outcome(rep),
                         "codes": ";".join(rep.codes(rp.ERROR) + rep.codes(rp.WARNING))})
    return pd.DataFrame(rows), reps, ids, site, cases


def first_flag(long):
    """For each fixture: the first method (in order) that flags it, and how."""
    out = {}
    for fid, g in long.groupby("fid", sort=False):
        g = g.set_index("method").reindex(METHODS)
        hit = g[g["outcome"] != "silent"]
        out[fid] = "none" if hit.empty else f"{hit.index[0]}: {'ERROR' if hit['outcome'].iloc[0] == 'ERROR' else 'WARNING'}"
    return out


# --------------------------------------------------------------------------- ranking simulation
NORMALISERS = {"none": None, "average rank": rp.rank_average, "argsort of argsort": rp.rank_double_argsort}
SCENARIOS = {  # name -> (kind, number of fallback studies as a function of n, round outputs?)
    "valid: continuous outputs": ("valid", lambda n: 0, False),
    "valid: outputs rounded to 2 dp": ("valid", lambda n: 0, True),
    "fault: 1 fallback study": ("fault", lambda n: 1, False),
    "fault: ~3% fallback (min 2)": ("fault", lambda n: _k(n, 0.03, 2), False),
    "fault: 25% fallback": ("fault", lambda n: _k(n, 0.25, 2), False),
    "fault: 100% fallback": ("fault", lambda n: n, False),
}


def rank_simulation(ns=(35, 100, 1300), seeds=range(50), base_seed=20261010):
    """Per run: random Beta(2,5) outputs, k studies replaced by the 0.5 fallback, then each
    normaliser; record whether the CSV-only guard and the raw-vs-final checks flag it."""
    rows = []
    for n in ns:
        ids = synthetic_ids(n, base_seed + n)
        for s in seeds:
            rng = np.random.default_rng([base_seed, n, s])
            base = rng.beta(2, 5, size=(n, len(LABELS)))
            for scen, (kind, k_of, rounded) in SCENARIOS.items():
                x = np.round(base, 2) if rounded else base.copy()
                k = k_of(n)
                if k:
                    x[rng.choice(n, size=k, replace=False)] = 0.5
                raw = pd.DataFrame(x, columns=LABELS)
                raw.insert(0, ID, ids)
                for norm, fn in NORMALISERS.items():
                    sub = raw.copy()
                    if fn is not None:
                        sub[LABELS] = fn(raw[LABELS])
                    g = rp.check_nonconstant(sub).codes()
                    r = rp.check_raw(raw, sub).codes()
                    rows.append({"n": n, "seed": s, "scenario": scen, "kind": kind, "k": k, "normaliser": norm,
                                 "guard_constant": "GUARD_CONSTANT_COLUMNS" in g,
                                 "guard_identical_rows": "GUARD_IDENTICAL_ROWS" in g,
                                 "csv_guard_any": bool(g),
                                 "raw_identical": bool({"RAW_IDENTICAL_VECTORS", "RAW_ALL_IDENTICAL"} & set(r)),
                                 "raw_ties_broken": "RAW_TIES_BROKEN" in r,
                                 "raw_any": bool(r)})
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- real DICOM helpers
UNCOMPRESSED = {"1.2.840.10008.1.2", "1.2.840.10008.1.2.1", "1.2.840.10008.1.2.2"}


def inference_view(root):
    """Training studies shaped exactly like the test-time interface.

    Only the columns that test.csv / test_series.csv provide are loaded, so the
    Report text and the labels never enter the inference code."""
    study_cols = list(pd.read_csv(root / "test.csv", nrows=0).columns)
    series_cols = list(pd.read_csv(root / "test_series.csv", nrows=0).columns)
    studies = pd.read_csv(root / "train.csv", usecols=study_cols)
    series = pd.read_csv(root / "train_series.csv", usecols=series_cols)
    banned = {"Report", *LABELS}
    assert not banned & set(studies.columns) and not banned & set(series.columns)
    return studies, series, study_cols, series_cols


def scan_syntax(root, series, study_ids):
    """Transfer syntax of the first file (sorted by name) of every series; header only."""
    import pydicom
    rows = []
    for sid in study_ids:
        for se in series.loc[series[ID] == sid, rp.SERIES_COL]:
            files = sorted((root / sid / se).glob("*.dcm"))
            ts = str(pydicom.dcmread(files[0], stop_before_pixels=True).file_meta.TransferSyntaxUID) if files else "none"
            rows.append({ID: sid, rp.SERIES_COL: se, "n_files": len(files), "transfer_syntax": ts,
                         "compressed": ts not in UNCOMPRESSED})
    return pd.DataFrame(rows)


def link_tree(src_root, dst_root, series, omit=(), truncate=()):
    """Overlay <dst>/<study>/<series> pointing at the read-only input.

    omit: (study, series) pairs left out. truncate: pairs whose files are copied and cut
    to half their size (a real damaged file, not a simulated exception)."""
    dst_root = Path(dst_root)
    shutil.rmtree(dst_root, ignore_errors=True)
    for sid, se in series[[ID, rp.SERIES_COL]].itertuples(index=False):
        if (sid, se) in omit:
            continue
        src, dst = Path(src_root) / sid / se, dst_root / sid / se
        dst.parent.mkdir(parents=True, exist_ok=True)
        if (sid, se) in truncate:
            dst.mkdir()
            for f in src.glob("*.dcm"):
                b = f.read_bytes()
                (dst / f.name).write_bytes(b[: len(b) // 2])
            continue
        try:
            os.symlink(src.resolve(), dst, target_is_directory=True)
        except OSError:
            shutil.copytree(src, dst)
    return dst_root


class RealSite:
    """Same interface as SyntheticSite, backed by real DICOM folders and the adapter reader."""

    def __init__(self, root, series):
        self.root, self.series = Path(root), series

    def reader(self, record_uid=True):
        import adapter_example as ae
        return ae.make_reader(self.root, self.series, record_uid=record_uid)

In [ ]:
%%writefile adapter_example.py
"""Runnable adapter example for rsna_preflight.

Replace `predict` with your model. Keep `read_study` strict: it must raise when a
series is missing, a file cannot be decoded, or the headers name another study,
so the failure is recorded instead of silently becoming a default prediction.

    python adapter_example.py --root <competition folder> --split test --out preflight_demo
"""
from __future__ import annotations

import argparse
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

import rsna_preflight as rp

COMPETITION_DIRS = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
                    "/kaggle/input/rsna-knee-abnormality-detection"]


def find_root(override=None):
    for d in ([override] if override else []) + [os.environ.get("RSNA_ROOT")] + COMPETITION_DIRS:
        if d and (Path(d) / "sample_submission.csv").is_file():
            return Path(d)
    raise FileNotFoundError("competition folder not found; pass --root or set RSNA_ROOT")


def make_reader(series_root, series_table=None, slices_per_series=3, record_uid=True):
    """Strict DICOM reader: listed series must exist, every chosen slice must decode.

    Reads all headers of a series (no pixels), orders by InstanceNumber and decodes
    `slices_per_series` slices from the middle. Records the StudyInstanceUID found in
    the headers so a prediction can be traced to the study it was computed from."""
    import pydicom

    root = Path(series_root)
    listed = {}
    if series_table is not None:
        for s, g in series_table.astype(str).groupby(rp.ID_COL):
            listed[s] = list(g[rp.SERIES_COL])

    def read_study(study_id: str) -> rp.StudyData:
        sdir = root / study_id
        series = listed.get(study_id) or sorted(os.listdir(sdir))
        images, uids = {}, set()
        for se in series:
            files = sorted((sdir / se).glob("*.dcm"))       # FileNotFoundError is not raised by glob
            if not files:
                raise FileNotFoundError(f"series {se[-12:]} has no readable .dcm files")
            heads = [pydicom.dcmread(f, stop_before_pixels=True) for f in files]
            order = np.argsort([int(getattr(h, "InstanceNumber", 0) or 0) for h in heads], kind="stable")
            mid = len(order) // 2
            pick = order[max(0, mid - slices_per_series // 2): mid - slices_per_series // 2 + slices_per_series]
            stack = []
            for i in pick:
                ds = pydicom.dcmread(files[i])
                uids.add(str(ds.StudyInstanceUID))
                stack.append(ds.pixel_array.astype(np.float32))   # raises if no decoder for the syntax
            images[se] = stack
        if len(uids) > 1:
            raise ValueError(f"headers name {len(uids)} different studies")
        return rp.StudyData(images=images, n_series=len(images),
                            n_images=sum(len(v) for v in images.values()),
                            observed_study_uid=(uids.pop() if uids and record_uid else None))

    return read_study


def demo_predict(data: rp.StudyData) -> np.ndarray:
    """PLACEHOLDER, not a model: 12 numbers from simple image statistics.

    It is deterministic and label-free so the harness has something to check.
    Its values carry no diagnostic meaning; never submit them."""
    feats = []
    for stack in data.images.values():
        a = np.stack([s / (np.percentile(s, 99) + 1e-6) for s in stack])
        feats += [a.mean(), a.std()]
    f = np.array(feats[:4] + [0.0] * max(0, 4 - len(feats)))
    w = np.random.default_rng(0).normal(size=(len(rp.LABELS), 4))
    return 1.0 / (1.0 + np.exp(-(w @ (f - 0.3))))


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--root")
    ap.add_argument("--split", default="test", choices=["test"])
    ap.add_argument("--out", default="preflight_demo")
    a = ap.parse_args(argv)
    root = find_root(a.root)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    ids = pd.read_csv(root / "test.csv")[rp.ID_COL].astype(str).tolist()
    series = pd.read_csv(root / "test_series.csv")
    t0 = time.perf_counter()
    res = rp.run_inference(ids, make_reader(root / "test_series", series), demo_predict,
                           rp.LABELS, fallback=0.5, normalize=rp.rank_average)
    inputs, _ = rp.check_inputs(root / "test_series", ids, series)
    rep = rp.preflight(res.submission, ids, raw=res.raw, provenance=res.provenance, inputs=inputs, fallback=0.5)
    rep.meta["seconds"] = round(time.perf_counter() - t0, 2)
    res.raw.to_csv(out / "raw_predictions.csv", index=False)
    res.provenance.to_csv(out / "provenance.csv", index=False)
    res.submission.to_csv(out / "demo_predictions_not_for_submission.csv", index=False)
    rep.to_json(out / "preflight_report.json")
    print(rep.summary())
    return rep


if __name__ == "__main__":
    main()

In [ ]:
import os, sys, time, json, hashlib, platform, importlib
from pathlib import Path
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

T0 = time.perf_counter()
SEED = 20261010
np.random.seed(SEED)
sys.path.insert(0, os.getcwd())
import rsna_preflight as rp, preflight_lab as L, adapter_example as ae
for m in (rp, L, ae):
    importlib.reload(m)
ID, LABELS = rp.ID_COL, rp.LABELS
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
RESULTS = {"seed": SEED}
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"
ERR, WARN, SILENT = "#d03b3b", "#fab219", "#e9e8e4"

## 1. The counterexample: 35 failed studies, 35 "healthy" ranks

*An injected failure, built on purpose to show a weakness in a check. It is not a diagnosis of any real submission.*

A *fallback* is the default value a pipeline writes when a study fails (here `0.5`). Suppose all 35 studies fail and each gets `0.5` for all 12 findings. Many pipelines rank-normalise before writing the CSV, then assert that no column is constant. Here are two common ways to rank, followed by that guard:

In [ ]:
# 35 studies whose inference failed: every one of the 12 outputs is the fallback 0.5.
failed = pd.DataFrame(np.full((35, len(LABELS)), 0.5), columns=LABELS)
failed.insert(0, ID, L.synthetic_ids(35, SEED))

# The two rank steps, written out (rp.rank_double_argsort / rp.rank_average):
#   np.argsort(np.argsort(x, axis=0), axis=0)          -> a permutation: ties broken by row position
#   pd.DataFrame(x).rank(method="average", pct=True)   -> tied inputs share one averaged rank
def ranked(fn):
    out = failed.copy(); out[LABELS] = fn(failed[LABELS]); return out
cx = {"argsort of argsort": ranked(rp.rank_double_argsort), "rank(method='average')": ranked(rp.rank_average)}

rows = []
for name, sub in cx.items():
    guard = rp.check_nonconstant(sub)                 # 'every column has >1 value' + 'no identical rows'
    raw = rp.check_raw(failed, sub)                   # needs the pre-rank predictions
    n = sub[LABELS].nunique()
    rows.append({"rank step": name,
                 "distinct values per label": f"{n.min()}" if n.min() == n.max() else f"{n.min()}-{n.max()}",
                 "CSV guard": ", ".join(guard.codes()) or "silent (looks healthy)",
                 "raw-vs-final check": ", ".join(raw.codes(rp.WARNING)) or "silent"})
counter = pd.DataFrame(rows)
display(counter)
RESULTS["counterexample"] = counter.to_dict("records")

fig, ax = plt.subplots(figsize=(8, 1.9))
for y, (name, sub) in enumerate(cx.items()):
    v = sub["ACL"].to_numpy()
    ax.scatter(v, np.full(len(v), y), s=26, color=[ERR, MUTED][y], edgecolor=SURFACE, linewidth=0.8, zorder=3)
    ax.text(1.03, y, f"{len(np.unique(v))} distinct", va="center", color=INK, fontsize=9)
ax.set_yticks([0, 1], ["argsort of argsort", "rank(average)"]); ax.set_ylim(-0.6, 1.6); ax.set_xlim(0, 1.18)
ax.set_xlabel("final ACL value written for the 35 failed studies"); ax.grid(axis="x", color=SILENT, lw=0.8)
ax.set_title("Same 35 identical fallbacks, two rank steps", loc="left", fontsize=11, color=INK)
plt.tight_layout(); plt.savefig(OUT / "counterexample.png", dpi=130); plt.show()

`argsort` returns a **permutation**, so argsort-of-argsort gives every row a different rank by construction. Ties are broken by row position. After that step the constant-column guard and the identical-row guard **can never fire**, whatever happened upstream. `rank(method="average")` gives tied inputs the same value, so both guards still see the failure.

The raw-vs-final check (section 2) catches it either way, because it compares the CSV with the predictions *before* ranking.

**Practical rule:** run value checks on the raw predictions, or rank with averaged ties.

## 2. Detection matrix: what each layer of evidence catches

Each column adds one kind of evidence to the column before it:

1. **CSV check** is what a basic validator does: schema, study IDs (missing, duplicate, foreign) and finite values.
2. **+ non-constant guard** adds "no constant column" and "no two studies with an identical 12-value row". Both are *warnings*, because constant predictions can be legitimate.
3. **+ raw predictions** adds the array captured just before any rank step, keyed by study and named in the model's own label order. It checks that every submitted study was predicted and finite, looks for identical vectors, and asks whether the submitted values are a **monotonic function** of the raw ones (a higher raw prediction never ends up lower in the CSV). That last check catches shifted rows, swapped labels and broken ties.
4. **+ provenance & inputs** adds one log record per study (status, fallback used, images read, `StudyInstanceUID` from the DICOM headers) and compares the input folder with the series list.

**ERROR** means a recorded contract failure. **WARN** means suspicious but possibly legitimate. The fixture list and each fixture's expected first flag were written down before the first run.

In [ ]:
fx = L.fixtures()
display(pd.DataFrame([{"id": f.fid, "type": f.kind, "fixture": f.name, "what is injected": f.injected,
                       "predeclared first flag": f.expect} for f in fx]).set_index("id"))
t = time.perf_counter()
long, reps, ids, site, cases = L.run_matrix("lab_site", 100, SEED)
RESULTS["matrix_seconds"] = round(time.perf_counter() - t, 2)
wide = long.pivot(index="fid", columns="method", values="outcome")[L.METHODS].reindex([f.fid for f in fx])
observed = L.first_flag(long)
norm = lambda s: s.replace(" (allowed)", "").split(" (needs")[0]
check = pd.DataFrame({"predeclared": {f.fid: f.expect for f in fx}, "observed": observed})
check["match"] = [norm(a) == norm(b) for a, b in zip(check.predeclared, check.observed)]
long.to_csv(OUT / "detection_matrix_long.csv", index=False)

fig, ax = plt.subplots(figsize=(9.6, 9.0))
names = {f.fid: f"{f.fid}  {f.name}" for f in fx}
col = {"ERROR": ERR, "WARN": WARN, "silent": SILENT}
txt = {"ERROR": ("ERROR", "white"), "WARN": ("WARN", INK), "silent": ("-", MUTED)}
nf = sum(f.kind == "fault" for f in fx)
ypos = [i + 1 if i < nf else i + 2.2 for i in range(len(wide))]          # room for two section headers
for y, fid in zip(ypos, wide.index):
    for j, m in enumerate(L.METHODS):
        o = wide.loc[fid, m]
        ax.add_patch(plt.Rectangle((j + 0.04, y + 0.06), 0.92, 0.88, color=col[o], lw=0))
        ax.text(j + 0.5, y + 0.5, txt[o][0], ha="center", va="center", color=txt[o][1], fontsize=8.5,
                fontweight="bold" if o != "silent" else "normal")
ax.text(0.04, 0.55, f"INJECTED FAULTS ({nf})", fontsize=9, fontweight="bold", color=INK, va="center")
ax.text(0.04, nf + 1.75, f"VALID CONTROLS ({len(wide) - nf}): an ERROR here would be a false alarm",
        fontsize=9, fontweight="bold", color=INK, va="center")
ax.set_xlim(0, len(L.METHODS)); ax.set_ylim(ypos[-1] + 1, 0)
ax.set_xticks(np.arange(len(L.METHODS)) + 0.5, L.METHODS); ax.xaxis.tick_top()
ax.set_yticks(np.array(ypos) + 0.5, [names[f] for f in wide.index])
ax.tick_params(length=0); [s.set_visible(False) for s in ax.spines.values()]
fig.suptitle("Which check flags which injected fault", x=0.02, ha="left", fontsize=12, color=INK)
fig.text(0.02, 0.945, "100 synthetic studies; each column adds one kind of evidence to the column before it",
         fontsize=9, color=MUTED)
plt.tight_layout(rect=(0, 0, 1, 0.93)); plt.savefig(OUT / "detection_matrix.png", dpi=130); plt.show()

faults, valid = wide.loc[[f.fid for f in fx if f.kind == "fault"]], wide.loc[[f.fid for f in fx if f.kind == "valid"]]
summary = pd.DataFrame({
    "faults flagged ERROR": (faults == "ERROR").sum(),
    "faults flagged at all": (faults != "silent").sum(),
    f"faults (of {len(faults)}) missed": (faults == "silent").sum(),
    f"valid controls (of {len(valid)}) with ERROR": (valid == "ERROR").sum(),
    "valid controls with WARN": (valid == "WARN").sum()}).T
display(summary)
RESULTS["matrix"] = {"wide": wide.to_dict(), "summary": summary.to_dict(), "predeclared_match": int(check["match"].sum()),
                     "n_fixtures": len(check)}
print("predeclared first-flag matches:", int(check["match"].sum()), "of", len(check))

**Reading the matrix**

- The CSV check sees only the 4 faults that damage the file itself (F01–F04).
- The guard adds 3 fallback cases as warnings, but only when ties survive ranking. With argsort-of-argsort (F09, F11) it stays silent.
- Raw predictions catch the NaN that `fillna` hid (F05), the partial run (F06), and rows or labels moved at export (F12, F13).
- Only the per-study log catches a single missing series (F07) and a prediction computed from the wrong study (F14).
- **F15 is missed by every layer, as predicted.** If the adapter records no header UID, a CSV, raw predictions and a status log cannot reveal that a prediction belongs to another study. A CSV alone never reveals semantically swapped predictions.
- Valid controls: shuffled-but-keyed rows, raw probabilities and rounded outputs raise **no errors**. The intended constant prior (V04) and broken legitimate ties (V06) raise **warnings only**.

In [ ]:
# What the full harness said, fixture by fixture (codes are the structured diagnostics)
full = long[long.method == L.METHODS[3]].set_index("fid")[["name", "outcome", "codes"]]
display(full.join(check[["predeclared", "match"]]))
reps["F09"][L.METHODS[3]].to_json(OUT / "example_report_F09.json")
print(reps["F09"][L.METHODS[3]].summary())

## 3. Ties kept vs ties broken, measured

The counterexample is one case. How often does each check catch a constant or partial fallback?

Setup: Beta(2,5) random outputs for 12 labels, with *k* studies replaced by the `0.5` fallback. Each table is then passed through no rank step, average rank, or argsort-of-argsort. This is repeated over 30 seeds at 35, 100 and 1,300 studies. The table shows the % of runs flagged.

In [ ]:
t = time.perf_counter()
sim = L.rank_simulation(ns=(35, 100, 1300), seeds=range(30), base_seed=SEED)
RESULTS["ranksim_seconds"] = round(time.perf_counter() - t, 2)
sim.to_csv(OUT / "rank_simulation.csv", index=False)
rate = (sim.groupby(["scenario", "normaliser"], sort=False)[["csv_guard_any", "raw_identical", "raw_ties_broken"]]
        .mean().mul(100).round(1))
tab = rate["csv_guard_any"].unstack()[list(L.NORMALISERS)]
tab.columns = [f"CSV guard, {c}" for c in tab.columns]
tab["raw: identical vectors"] = rate["raw_identical"].groupby(level=0, sort=False).max()
tab["raw: ties broken (argsort of argsort)"] = rate["raw_ties_broken"].xs("argsort of argsort", level=1)
tab = tab.reindex(list(L.SCENARIOS))
display(tab.style.format("{:.0f}%").background_gradient(cmap="Greys", vmin=0, vmax=250))
n_runs = sim.groupby(["scenario", "normaliser"]).size().iloc[0]
by_n = sim.groupby(["n", "scenario", "normaliser"])["csv_guard_any"].mean().unstack()
same_across_n = bool((by_n.groupby(level=1).nunique() <= 1).all().all())
print("runs per cell:", n_runs, "| identical rates at n=35/100/1300:", same_across_n)
RESULTS["ranksim"] = {"table_pct": tab.to_dict(), "runs_per_cell": int(n_runs), "same_across_n": same_across_n}

- **Argsort-of-argsort: the CSV-only guard flagged 0% of fallback runs at every size tested.** With no rank step or with average rank it flagged 100% of runs with two or more fallback studies.
- **One fallback study: 0% for every check that looks at values,** under all three rank steps. A single `0.5` row looks like any other prediction. Only a recorded failure shows it.
- The raw identical-vector check finds 2+ fallback studies whatever the rank step.
- **Cost of the raw tie check:** it also warns on 100% of *valid* runs whose outputs were rounded and then ranked by argsort-of-argsort, because legitimate ties get broken too. That is why it is a warning, not an error.
- Rates were the same at 35, 100 and 1,300 studies. Most cells are 0% or 100% by construction, so the seeds confirm determinism; they do not estimate a probability.

## 4. Smoke test on real DICOMs

The reader is the part most likely to fail on unseen files, so it is tested on official **training** studies used as unlabeled inputs. The interface is built from exactly the columns that `test.csv` and `test_series.csv` provide, so the `Report` text and the labels are never loaded. Studies come from a seeded, bounded sample. Faults are injected through an overlay folder of links, and the read-only input is never modified. The predictor is a **placeholder** (image statistics, not a model), so these outputs mean nothing for AUC.

In [ ]:
ROOT = None
try:
    ROOT = ae.find_root()
except FileNotFoundError as e:
    print("competition data not attached:", e)
if ROOT is not None:
    studies, series, study_cols, series_cols = L.inference_view(ROOT)   # Report + labels are never loaded
    print("study columns: ", ", ".join(study_cols))
    print("series columns:", ", ".join(series_cols))
    print(f"unlabeled training inputs: {len(studies)} studies, {len(series)} series")
    RESULTS["view"] = {"study_cols": study_cols, "series_cols": series_cols, "n_studies": len(studies)}

In [ ]:
if ROOT is not None:
    t = time.perf_counter()
    order = list(np.random.default_rng(SEED).permutation(studies[ID].astype(str).to_numpy()))
    scans, plain, packed = [], [], []
    for sid in order[:200]:                                   # bounded header scan, seeded order
        sc = L.scan_syntax(ROOT / "train_series", series, [sid]); scans.append(sc)
        (packed if sc["compressed"].any() else plain).append(sid)
        if len(scans) >= 60 and len(packed) >= 2:
            break
    scan = pd.concat(scans, ignore_index=True)
    syntax_counts = scan.groupby("transfer_syntax").agg(series=(ID, "size"), studies=(ID, "nunique"))
    display(syntax_counts)
    cohort, extra = plain[:6], packed[:2]
    print(f"scanned: {scan[ID].nunique()} studies, {len(scan)} series")
    print(f"studies with a compressed first file: {len(packed)}" + ("" if packed else " (S2 cannot run)"))
    cs = series[series[ID].isin(cohort + extra)].astype(str)
    smoke_rows, smoke_reps, smoke_cases = [], {}, {}
    def smoke(name, ids, omit=(), truncate=(), **kw):
        root = L.link_tree(ROOT / "train_series", "overlay", cs[cs[ID].isin(ids)], omit=omit, truncate=truncate)
        site = L.RealSite(root, cs[cs[ID].isin(ids)])
        tt = time.perf_counter()
        case = L.user_loop(site, ids, predict=ae.demo_predict, **kw)
        reps_ = L.method_reports(case, ids, site); smoke_reps[name] = reps_; smoke_cases[name] = case
        row = {"scenario": name, "studies": len(ids), "seconds": round(time.perf_counter() - tt, 2)}
        row.update({m: L.outcome(r) for m, r in reps_.items()})
        row["codes (full harness)"] = ";".join(reps_[L.METHODS[3]].codes(rp.ERROR) + reps_[L.METHODS[3]].codes(rp.WARNING))
        smoke_rows.append(row)
    first = cs[cs[ID] == cohort[0]][rp.SERIES_COL].tolist()
    smoke("S1 clean control (uncompressed studies)", cohort)
    if extra:
        smoke("S2 + studies with compressed syntax (real decoder availability)", cohort + extra)
    smoke("S3 one listed series folder missing", cohort, omit={(cohort[0], first[1])})
    smoke("S4 one series of damaged files (truncated copies)", cohort, truncate={(cohort[0], first[0])})
    smoke("S5 next study's folder opened, header UID recorded", cohort, open_ids=cohort[1:] + cohort[:1])
    smoke("S6 same bug, adapter records no header UID", cohort, open_ids=cohort[1:] + cohort[:1], record_uid=False)
    smoke_df = pd.DataFrame(smoke_rows).set_index("scenario")
    display(smoke_df)
    prov1 = smoke_cases["S1 clean control (uncompressed studies)"]["prov"]
    uid_eq = bool((prov1["observed_study_uid"].astype(str) == prov1["study_id"]).all())
    print(f"S1: {int(prov1['n_series'].sum())} series, {int(prov1['n_images'].sum())} slices decoded")
    print("S1: header StudyInstanceUID == folder name for every study:", uid_eq)
    RESULTS["smoke"] = {"table": smoke_df.reset_index().to_dict("records"), "scanned_studies": int(scan[ID].nunique()),
                        "scanned_series": len(scan), "syntax_counts": syntax_counts.reset_index().to_dict("records"),
                        "compressed_studies_found": len(packed), "seconds": round(time.perf_counter() - t, 2),
                        "s1_series": int(prov1["n_series"].sum()), "s1_images": int(prov1["n_images"].sum()), "s1_uid_equal": uid_eq}
    smoke_df.to_csv(OUT / "dicom_smoke.csv"); scan.to_csv(OUT / "syntax_scan.csv", index=False)
    for k in [x for x in smoke_reps if x.startswith(("S2", "S4"))]:
        print(k); display(smoke_reps[k][L.METHODS[3]].table()[["code", "examples"]].head(3))

**On the real files (Kaggle CPU run):**

- **Clean control (S1).** All layers stayed silent on 6 training studies (28 series, 84 decoded slices). The strict reader decoded every chosen slice, and each header `StudyInstanceUID` equalled its folder name, so the UID check raised no false alarm on real data.
- **S3, S4, S5 are invisible to the CSV, the guard and the raw predictions,** because each failure becomes one ordinary-looking fallback row or a wrong-but-plausible row. The per-study record catches every one:
  - a missing series folder, also flagged by the input audit;
  - a real damaged file, where pydicom raised "number of bytes of pixel data is less than expected";
  - a prediction computed from the next study's folder.
- **S6 is missed, as F15 predicted.** Without the header UID nothing can show the wrong-study read.
- **Transfer syntaxes:** the first file of every series in the seeded scan was uncompressed explicit-VR little endian. No compressed file was found, so the compressed-codec scenario did not run, and the codec fixture in section 2 stays *simulated*. This image has pydicom but no pylibjpeg, openjpeg or gdcm. Whether it decodes the compressed syntaxes mentioned on the data page was **not tested** here.
- The adapter example also ran on the 3 visible test studies and passed every check.

In [ ]:
if ROOT is not None:
    demo = ae.main(["--root", str(ROOT), "--out", str(OUT / "test_demo")])     # the 3 visible test studies
    if demo.findings:
        display(demo.table())
    RESULTS["test_demo"] = {"status": demo.status, "codes": demo.codes(), "seconds": demo.meta.get("seconds")}

## 5. Bookkeeping at 3, 100 and 1,300 studies

3 is the visible test size and ~1,300 the stated hidden size. The full fixture set is rerun at each size with three seeded input orders, using the synthetic stub reader. **This tests ID bookkeeping only, not GPU memory or your model's hidden-test runtime.**

In [ ]:
scale_rows, perm_diff = [], []
base = wide                                                     # the N=100 matrix above
for n in (3, 100, 1300):
    outs = {}
    for p in (0, 1, 2):
        t = time.perf_counter()
        lg, _, ids_p, _, cs_p = L.run_matrix("lab_site_scale", n, SEED + n, perm_seed=p)
        sec = time.perf_counter() - t
        w = lg.pivot(index="fid", columns="method", values="outcome")[L.METHODS].reindex(base.index)
        valid_err = int((w.loc[[f.fid for f in fx if f.kind == "valid"]] == "ERROR").sum().sum())
        missed = int((w.loc[[f.fid for f in fx if f.kind == "fault"], L.METHODS[3]] == "silent").sum())
        scale_rows.append({"studies": n, "permutation seed": p, "cells equal to N=100 matrix": int((w == base).sum().sum()),
                           "cells": w.size, "valid-control ERRORs": valid_err, "faults missed by full harness": missed,
                           "seconds (21 fixtures)": round(sec, 2)})
        outs[p] = {k: cs_p[k]["sub"].sort_values(ID).reset_index(drop=True) for k in ("V01", "V06")}
    for k in ("V01", "V06"):
        a, b = outs[0][k][LABELS].to_numpy(), outs[1][k][LABELS].to_numpy()
        perm_diff.append({"studies": n, "fixture": k, "rank step": "average" if k == "V01" else "argsort of argsort",
                          "cells that change when input order changes": f"{(a != b).mean() * 100:.1f}%"})
scale = pd.DataFrame(scale_rows); pdiff = pd.DataFrame(perm_diff)
display(scale); display(pdiff)
scale.to_csv(OUT / "scale_bookkeeping.csv", index=False)
RESULTS["scale"] = {"rows": scale.to_dict("records"), "perm_diff": pdiff.to_dict("records")}

- **Same outcomes at 100 and 1,300 studies** for all three input orders: 84 of 84 matrix cells match section 2.
- **One cell differs at 3 studies.** F06's partial run leaves a single pre-filled row there, so the identical-row guard has no tie to see. Raw predictions and provenance still raise ERROR.
- **0 valid-control ERRORs** at every size and order. The full harness missed only F15, as designed.
- **Average rank does not depend on file order:** 0% of cells changed between two input orders. **Argsort-of-argsort does** when outputs are legitimately tied (rounded): 5.6%, 38.7% and 92.5% of cells changed at 3, 100 and 1,300 studies. The same predictions in a different `os.listdir` order give a different submission.
- Timings for all 21 fixtures are in the table above. They measure bookkeeping on a stub, not your model.

## 6. Plug in your pipeline

**What the checker needs from you:**

| Evidence | What to record | Unlocks |
|---|---|---|
| submission + `test.csv` | nothing extra | CSV layer |
| `raw` | the last array **before** any rank or normalisation step, columns named in *your model's* label order | NaN hidden by `fillna`, partial runs, shifted rows, swapped labels, broken ties |
| `ProvenanceLog` | one `record()` per study: status, `fallback=True` when you write a default, images read, `StudyInstanceUID` from the headers you actually read | single failed studies, wrong-study reads |
| input audit | `check_inputs(series_root, ids, series_csv)` | missing or empty series folders |

```python
import rsna_preflight as rp
log, rows = rp.ProvenanceLog(), []
for sid in test_ids:
    try:
        x, uid, n = my_reader(sid)            # must RAISE on missing/undecodable files
        y = my_model(x)                       # 12 probabilities, in MY_LABEL_ORDER
        log.record(sid, n_images=n, observed_study_uid=uid)
    except Exception as e:
        y = [0.5] * 12
        log.record(sid, status="error", fallback=True, error=repr(e))
    rows.append([sid, *y])
raw = pd.DataFrame(rows, columns=[rp.ID_COL, *MY_LABEL_ORDER])     # capture BEFORE ranking
sub = raw[[rp.ID_COL, *rp.LABELS]].copy()
sub[rp.LABELS] = rp.rank_average(sub[rp.LABELS])                    # ties kept
rep = rp.preflight(sub, test_ids, raw=raw, provenance=log.frame())
rep.to_json("preflight_report.json"); print(rep.summary())
```

`adapter_example.py` is a complete runnable version: a strict pydicom reader plus the reference loop `rp.run_inference`. It is also available from the command line: `python rsna_preflight.py --submission submission.csv --expected test.csv --raw raw.csv --provenance provenance.csv --json report.json` (exits 1 on any ERROR).

**On the hidden rerun you cannot read this report.** You choose a policy. Raising on ERROR (`if rep.errors: raise RuntimeError(rep.summary())`) turns a silent bad score into a visible failed submission. Writing the file anyway keeps the score but hides the failure. The checker makes that choice explicit; it cannot make it for you.

## 7. What this cannot detect, and limitations

**Cannot detect:**
- A wrong model, wrong preprocessing or poor predictions that are correctly keyed. These are AUC problems, not contract failures.
- Wrong-study reads when no header UID is recorded (F15).
- Label-order errors when `raw` is already named in the wrong order.
- Failures your loop never records.
- Anything that happens after `raw` is captured, if you blend or edit afterwards.
- GPU out-of-memory, timeouts, platform or queue failures, and hidden-test-only file properties absent from the visible sample.

**Limitations:**
- The checks and the fixtures were written by the same author. The matrix shows what each *kind of evidence* can reveal; it is not an independent benchmark of the tool.
- Fixtures are deliberate, simple faults: the matrix measures *these* faults, not real-world failure frequency.
- Codec failures in the synthetic fixtures are simulated. The real-DICOM section shows what actually happens in this image.
- The real-DICOM sample is small and seeded. It is not a survey of the dataset.
- Warnings need human judgement. A constant prior or rounded outputs can be legitimate.
- Passing every check does not mean the hidden rerun will succeed or score well.

In [ ]:
vers = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__, "rsna_preflight": rp.__version__}
try:
    import pydicom; vers["pydicom"] = pydicom.__version__
except ImportError:
    vers["pydicom"] = "missing"
for m in ("pylibjpeg", "openjpeg", "gdcm"):
    vers[m] = "installed" if importlib.util.find_spec(m) else "not installed"
inputs = {}
if ROOT is not None:
    for f in ("sample_submission.csv", "test.csv", "test_series.csv", "train.csv", "train_series.csv"):
        b = (ROOT / f).read_bytes(); inputs[f] = {"bytes": len(b), "sha256": hashlib.sha256(b).hexdigest()[:16]}
try:
    import resource; peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
except ImportError:
    import psutil; peak = psutil.Process().memory_info().peak_wset / 2**20
RESULTS.update(versions=vers, inputs=inputs, peak_rss_mib=round(peak, 1), total_seconds=round(time.perf_counter() - T0, 1))
display(pd.Series(vers, name="version").to_frame()); display(pd.DataFrame(inputs).T)
print(f"total notebook time {RESULTS['total_seconds']} s | peak process RSS {RESULTS['peak_rss_mib']} MiB")
for f in ("rsna_preflight.py", "adapter_example.py", "preflight_lab.py"):
    print(f, hashlib.sha256(Path(f).read_bytes()).hexdigest()[:16])
(OUT / "run_summary.json").write_text(json.dumps(RESULTS, indent=1, default=str))

In [ ]:
# machine-readable copy of every result (also saved as outputs/run_summary.json); output hidden
print("RESULTS_JSON_BEGIN"); print(json.dumps(RESULTS, default=str)); print("RESULTS_JSON_END")

---
**Context (no code reused):** the closest public notebooks are a [submission validator](https://www.kaggle.com/code/beicicc/rsna-knee-submission-validator), which checks the CSV; a [template](https://www.kaggle.com/code/kiyoshiohno/rsna-knee-template), which runs an attached implementation; and [Fast 2xT4 Inference](https://www.kaggle.com/code/jiweiliu/rsna-knee-fast-2xt4-inference), which has its own runtime guards. This notebook instead measures detection limits on injected faults and adds an adapter-based harness. Data: [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data), used under the competition rules. Only training DICOM pixels and the test-shaped CSV columns are read.